In [ ]:
# @title 🧠 Dependencies { display-mode: "form" }
!pip uninstall -y google-adk bigframes opentelemetry-exporter-gcp-logging wandb
!pip install -q "crewai[tools]" "langchain-google-genai" "requests==2.32.4" "langchain" "langchain-openrouter" "yfinance" "pandas" "numpy"
print("✅ Clean CrewAI + Yahoo Finance environment installed.")

Found existing installation: google-adk 1.29.0
Uninstalling google-adk-1.29.0:
  Successfully uninstalled google-adk-1.29.0
Found existing installation: bigframes 2.42.0
Uninstalling bigframes-2.42.0:
  Successfully uninstalled bigframes-2.42.0
Found existing installation: opentelemetry-exporter-gcp-logging 1.11.0a0
Uninstalling opentelemetry-exporter-gcp-logging-1.11.0a0:
  Successfully uninstalled opentelemetry-exporter-gcp-logging-1.11.0a0
Found existing installation: wandb 0.27.2
Uninstalling wandb-0.27.2:
  Successfully uninstalled wandb-0.27.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# @title 🧠 Executive Intelligence Control Center { display-mode: "form" }
import os
import json
import warnings
import requests
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
warnings.filterwarnings("ignore")

from google.colab import userdata

SERPER_API_KEY = userdata.get("SERPER_API_KEY") or ""
TELEGRAM_BOT_TOKEN = userdata.get("TELEGRAM_BOT_TOKEN") or ""
TELEGRAM_CHAT_ID = userdata.get("TELEGRAM_CHAT_ID") or ""
OPENROUTER_API_KEY = userdata.get("OPENWARREN") or ""

TICKERS = ["NVO", "BKR", "GEV","GOOG"]
BENCHMARK = "^GSPC"
HISTORY_PERIOD = "5y"
NEWS_ITEMS_PER_TICKER = 5
EXPORT_DIR = "/content/output"
OPENROUTER_MODEL = "openrouter/owl-alpha"
MAX_TELEGRAM_CHARS = 3900
SEND_TELEGRAM_ON_START = True
SEND_TELEGRAM_FINAL_SUMMARY = True

os.environ["SERPER_API_KEY"] = SERPER_API_KEY
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["CREWAI_DISABLE_TRACING"] = "true"
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

OUTDIR = Path(EXPORT_DIR)
OUTDIR.mkdir(exist_ok=True)
AS_OF = pd.Timestamp.utcnow().strftime("%Y-%m-%d")

def escape_markdown_v2_final(text):
    escaped_text = text.replace("\\", "\\\\")
    for char in r'_*[]()~`>#+-=|{}.!':
        escaped_text = escaped_text.replace(char, "\\" + char)
    return escaped_text

def send_to_phone(text):
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        print("❌ Telegram sender not configured.")
        return None
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": escape_markdown_v2_final(str(text)[:MAX_TELEGRAM_CHARS]),
        "parse_mode": "MarkdownV2",
        "disable_web_page_preview": True,
    }
    try:
        response = requests.post(url, json=payload, timeout=30)
        print("Telegram status:", response.status_code)
        return response
    except Exception as e:
        print(f"❌ Telegram Network Error: {e}")
        return None

def chunk_text(text, max_len=3900):
    text = str(text)
    if len(text) <= max_len:
        return [text]
    chunks, current = [], ""
    for line in text.splitlines(True):
        if len(current) + len(line) <= max_len:
            current += line
        else:
            if current:
                chunks.append(current)
            current = line
    if current:
        chunks.append(current)
    return chunks

print("✅ Configuration loaded and Telegram sender ready.")

✅ Configuration loaded and Telegram sender ready.


In [ ]:
# @title 💹 Financial Data Extraction + Analyst Metrics { display-mode: "form" }
def safe_df(obj):
    try:
        if obj is None:
            return pd.DataFrame()
        if isinstance(obj, pd.DataFrame):
            return obj.copy()
        return pd.DataFrame(obj)
    except Exception:
        return pd.DataFrame()

def classify_ticker(ticker, info):
    qtype = str((info or {}).get("quoteType", "")).upper()
    symbol = str(ticker).upper()
    category = str((info or {}).get("category", "")).upper()
    long_name = str((info or {}).get("longName", "")).upper()

    if symbol.startswith("^") or qtype in {"INDEX"}:
        return "benchmark"
    if qtype in {"ETF", "MUTUALFUND", "MONEYMARKET", "FUND"}:
        return "fund"
    if qtype in {"BOND"} or "BOND" in category or "TREASURY" in long_name:
        return "bond"
    if qtype in {"CURRENCY", "CURRENCYPAIR", "FX"} or "=X" in symbol:
        return "fx"
    if qtype in {"CRYPTOCURRENCY", "CRYPTO"} or "-USD" in symbol and any(x in symbol for x in ["BTC", "ETH", "SOL", "XRP", "ADA"]):
        return "crypto"
    if qtype in {"FUTURE"} or "=F" in symbol:
        return "future"
    if qtype in {"OPTION"}:
        return "option"
    if qtype in {"EQUITY", "STOCK"}:
        return "equity"
    return "equity"

def get_first_available(series, keys):
    for key in keys:
        value = series.get(key)
        if value is not None and not (isinstance(value, float) and pd.isna(value)):
            return value
    return None

def to_dict_safe(df):
    out = safe_df(df)
    return {} if out.empty else out.replace({np.nan: None}).to_dict()

def to_records(df, n=None):
    out = safe_df(df)
    if n is not None:
        out = out.head(n)
    return [] if out.empty else out.replace({np.nan: None}).to_dict(orient="records")

def normalize_news(items, n=5):
    if not isinstance(items, list):
        return []
    cleaned = []
    for item in items[:n]:
        if isinstance(item, dict):
            cleaned.append({
                "title": item.get("title"),
                "publisher": item.get("publisher"),
                "link": item.get("link") or item.get("url"),
                "providerPublishTime": item.get("providerPublishTime"),
            })
    return cleaned

def fetch_ticker_bundle(ticker):
    tk = yf.Ticker(ticker)
    missing = []
    bundle = {"ticker": ticker}
    fields = {
        "info": lambda: tk.info,
        "fast_info": lambda: dict(tk.fast_info),
        "history": lambda: tk.history(period=HISTORY_PERIOD, auto_adjust=False),
        "income_stmt": lambda: tk.income_stmt,
        "quarterly_income_stmt": lambda: tk.quarterly_income_stmt,
        "balance_sheet": lambda: tk.balance_sheet,
        "quarterly_balance_sheet": lambda: tk.quarterly_balance_sheet,
        "cashflow": lambda: tk.cashflow,
        "quarterly_cashflow": lambda: tk.quarterly_cashflow,
        "earnings_dates": lambda: tk.get_earnings_dates(limit=8),
        "recommendations": lambda: tk.recommendations,
        "recommendations_summary": lambda: tk.recommendations_summary,
        "upgrades_downgrades": lambda: tk.upgrades_downgrades,
        "analyst_price_targets": lambda: tk.analyst_price_targets,
        "earnings_estimate": lambda: tk.earnings_estimate,
        "revenue_estimate": lambda: tk.revenue_estimate,
        "eps_trend": lambda: tk.eps_trend,
        "eps_revisions": lambda: tk.eps_revisions,
        "growth_estimates": lambda: tk.growth_estimates,
        "major_holders": lambda: tk.major_holders,
        "institutional_holders": lambda: tk.institutional_holders,
        "mutualfund_holders": lambda: tk.mutualfund_holders,
        "insider_transactions": lambda: tk.insider_transactions,
        "insider_purchases": lambda: tk.insider_purchases,
        "insider_roster_holders": lambda: tk.insider_roster_holders,
        "calendar": lambda: tk.calendar,
        "news": lambda: tk.news,
        "sec_filings": lambda: tk.sec_filings,
        "options": lambda: tk.options,
    }
    for name, fn in fields.items():
        try:
            bundle[name] = fn()
        except Exception as e:
            bundle[name] = None
            missing.append(f"{name}: {e.__class__.__name__}")
    bundle["data_quality"] = {"missing_modules": missing}
    return bundle

def compute_market_metrics(hist, benchmark_hist=None):
    hist = safe_df(hist)
    if hist.empty or "Close" not in hist:
        return {}
    px = hist["Close"].dropna()
    if px.empty:
        return {}
    rets = px.pct_change().dropna()
    out = {
        "price": float(px.iloc[-1]),
        "returns": {
            "1m": float(px.iloc[-1] / px.iloc[-22] - 1) if len(px) > 21 else None,
            "3m": float(px.iloc[-1] / px.iloc[-64] - 1) if len(px) > 63 else None,
            "6m": float(px.iloc[-1] / px.iloc[-127] - 1) if len(px) > 126 else None,
            "1y": float(px.iloc[-1] / px.iloc[-253] - 1) if len(px) > 252 else None,
        },
        "volatility_90d": float(rets.tail(90).std() * np.sqrt(252)) if len(rets) >= 20 else None,
        "max_drawdown_1y": float((px.tail(252) / px.tail(252).cummax() - 1).min()) if len(px) >= 20 else None,
    }
    benchmark_hist = safe_df(benchmark_hist)
    if not benchmark_hist.empty and "Close" in benchmark_hist and out["returns"]["1y"] is not None:
        bpx = benchmark_hist["Close"].dropna()
        if len(bpx) > 252:
            benchmark_1y = float(bpx.iloc[-1] / bpx.iloc[-253] - 1)
            out["returns"]["vs_spx_1y"] = out["returns"]["1y"] - benchmark_1y
    return out

def compute_fundamental_metrics(bundle):
    info = bundle.get("info") or {}
    inc = safe_df(bundle.get("income_stmt"))
    bal = safe_df(bundle.get("balance_sheet"))
    cf = safe_df(bundle.get("cashflow"))

    inc_l = inc.iloc[:, 0] if not inc.empty else pd.Series(dtype=float)
    bal_l = bal.iloc[:, 0] if not bal.empty else pd.Series(dtype=float)
    cf_l = cf.iloc[:, 0] if not cf.empty else pd.Series(dtype=float)

    revenue = get_first_available(inc_l, ["Total Revenue", "Operating Revenue"])
    gross_profit = get_first_available(inc_l, ["Gross Profit"])
    op_income = get_first_available(inc_l, ["Operating Income", "EBIT"])
    net_income = get_first_available(inc_l, ["Net Income"])
    cfo = get_first_available(cf_l, ["Operating Cash Flow", "Cash Flow From Continuing Operating Activities"])
    capex = get_first_available(cf_l, ["Capital Expenditure"])
    cash = get_first_available(bal_l, ["Cash And Cash Equivalents", "Cash Cash Equivalents And Short Term Investments"])
    debt = get_first_available(bal_l, ["Total Debt"])
    ebitda = info.get("ebitda")
    enterprise_value = info.get("enterpriseValue")
    market_cap = info.get("marketCap")
    equity = get_first_available(bal_l, ["Stockholders Equity", "Total Equity Gross Minority Interest"])

    fcf = None if cfo is None or capex is None else cfo + capex

    return {
        "revenue": {"latest": revenue},
        "margins": {
            "gross": None if revenue in (None, 0) or gross_profit is None else gross_profit / revenue,
            "operating": None if revenue in (None, 0) or op_income is None else op_income / revenue,
            "net": None if revenue in (None, 0) or net_income is None else net_income / revenue,
            "fcf": None if revenue in (None, 0) or fcf is None else fcf / revenue,
        },
        "cash_flow": {"cfo": cfo, "capex": capex, "fcf": fcf},
        "balance_sheet": {"cash": cash, "debt": debt, "net_debt": None if cash is None or debt is None else debt - cash},
        "returns": {"roe": None if equity in (None, 0) or net_income is None else net_income / equity},
        "valuation_derived": {
            "ev_sales": None if enterprise_value in (None, 0) or revenue in (None, 0) else enterprise_value / revenue,
            "ev_ebitda": None if enterprise_value in (None, 0) or ebitda in (None, 0) else enterprise_value / ebitda,
            "p_fcf": None if market_cap in (None, 0) or fcf in (None, 0) else market_cap / fcf,
        },
    }

def extract_next_earnings_date(earnings_dates_df, calendar_obj):
    edf = safe_df(earnings_dates_df)
    if not edf.empty:
        try:
            return str(pd.Timestamp(edf.index[0]).date())
        except Exception:
            pass
    cdf = safe_df(calendar_obj)
    if not cdf.empty:
        try:
            for col in cdf.columns:
                val = cdf[col].iloc[0]
                if pd.notna(val):
                    return str(pd.Timestamp(val).date())
        except Exception:
            pass
    return None

def build_packet(bundle, benchmark_hist=None):
    info = bundle.get("info") or {}
    asset_type = classify_ticker(bundle["ticker"], info)
    market = compute_market_metrics(bundle.get("history"), benchmark_hist)
    fundamentals = {} if asset_type == "benchmark" else compute_fundamental_metrics(bundle)
    targets = bundle.get("analyst_price_targets") or {}
    target_mean = targets.get("mean") if isinstance(targets, dict) else None
    current_price = market.get("price")
    packet = {
        "ticker": bundle["ticker"],
        "as_of": AS_OF,
        "asset_type": asset_type,
        "profile": {
            "company_name": info.get("longName") or info.get("shortName"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "country": info.get("country"),
        },
        "market": {**market, "market_cap": info.get("marketCap"), "currency": info.get("currency")},
        "fundamentals": fundamentals,
        "valuation": {
            "pe_ttm": info.get("trailingPE"),
            "pe_fwd": info.get("forwardPE"),
            "ev_sales": info.get("enterpriseToRevenue") or fundamentals.get("valuation_derived", {}).get("ev_sales"),
            "ev_ebitda": info.get("enterpriseToEbitda") or fundamentals.get("valuation_derived", {}).get("ev_ebitda"),
            "p_fcf": fundamentals.get("valuation_derived", {}).get("p_fcf"),
            "target_price_mean": target_mean,
            "target_upside": None if current_price in (None, 0) or target_mean is None else target_mean / current_price - 1,
        },
        "expectations": {
            "earnings_estimates": to_dict_safe(bundle.get("earnings_estimate")),
            "revenue_estimates": to_dict_safe(bundle.get("revenue_estimate")),
            "eps_revisions": to_dict_safe(bundle.get("eps_revisions")),
            "recommendations_summary": to_dict_safe(bundle.get("recommendations_summary")),
        },
        "ownership": {
            "major_holders": to_records(bundle.get("major_holders")),
            "institutional_holders": to_records(bundle.get("institutional_holders"), n=10),
            "insider_transactions": to_records(bundle.get("insider_transactions"), n=10),
        },
        "events": {
            "next_earnings_date": extract_next_earnings_date(bundle.get("earnings_dates"), bundle.get("calendar")),
            "recent_news": normalize_news(bundle.get("news"), NEWS_ITEMS_PER_TICKER),
        },
        "risk_flags": [],
        "data_quality": bundle.get("data_quality", {}),
    }
    return packet

def create_agent_input(packet):
    return {
        "ticker": packet["ticker"],
        "asset_type": packet["asset_type"],
        "company_name": packet.get("profile", {}).get("company_name"),
        "profile": packet.get("profile", {}),
        "market": packet.get("market", {}),
        "fundamentals": packet.get("fundamentals", {}),
        "valuation": packet.get("valuation", {}),
        "expectations": packet.get("expectations", {}),
        "ownership": packet.get("ownership", {}),
        "events": packet.get("events", {}),
        "risk_flags": packet.get("risk_flags", []),
        "data_quality": packet.get("data_quality", {}),
    }

if SEND_TELEGRAM_ON_START:
    send_to_phone(f"🚀 Financial pipeline started for: {', '.join(TICKERS)}")

bundles = {ticker: fetch_ticker_bundle(ticker) for ticker in TICKERS}
benchmark_hist = safe_df(bundles.get(BENCHMARK, {}).get("history")) if BENCHMARK in bundles else pd.DataFrame()
packets = {ticker: build_packet(bundle, benchmark_hist=benchmark_hist) for ticker, bundle in bundles.items()}
agent_inputs = {ticker: create_agent_input(packet) for ticker, packet in packets.items()}

with open(OUTDIR / "all_ticker_packets.json", "w") as f:
    json.dump(packets, f, indent=2, default=str)
with open(OUTDIR / "all_agent_inputs.json", "w") as f:
    json.dump(agent_inputs, f, indent=2, default=str)

print("✅ Financial data pulled and agent payloads ready.")

Telegram status: 200
✅ Financial data pulled and agent payloads ready.


In [ ]:
# @title 🧠 AGENTS SWARM { display-mode: "form" }
import warnings
warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew, Process
from crewai_tools import SerperDevTool
from langchain_openrouter import ChatOpenRouter

send_to_phone("🚀 Intelligence Matrix Initialized: starting financial analyst swarm.")

search_tool = SerperDevTool()
openrouter_llm = ChatOpenRouter(
    model_name=OPENROUTER_MODEL,
    openrouter_api_key=os.environ["OPENROUTER_API_KEY"],
    verbose=True,
)

index_tickers = [
    t for t in TICKERS
    if str(t).startswith("^") or agent_inputs.get(t, {}).get("asset_type") == "benchmark"
]
focus_tickers = [t for t in TICKERS if t not in index_tickers]

swarm_payload = json.dumps(
    {
        "as_of": AS_OF,
        "benchmark": BENCHMARK,
        "focus_tickers": focus_tickers,
        "index_tickers": index_tickers,
        "packets": {ticker: agent_inputs[ticker] for ticker in focus_tickers},
        "index_packets": {ticker: agent_inputs[ticker] for ticker in index_tickers},
        "benchmark_packet": agent_inputs.get(BENCHMARK, {}),
    },
    indent=2,
    default=str,
)

researcher = Agent(
    role="Senior Equity Research Analyst",
    goal="Review the prepared financial packets and identify the most important facts, signals, and current context for each ticker.",
    backstory=(
        "You think with Warren Buffett's temperament: patient, rational, business-first, valuation-disciplined, "
        "and focused on durable competitive advantage, capital allocation, intrinsic value, margin of safety, "
        "and downside protection. You avoid hype, short-term noise, and prediction theater. "
        "When enriching with web data, prefer recent information from the last 14 days and avoid sources older than 30 days unless clearly necessary."
    ),
    tools=[search_tool],
    llm=openrouter_llm,
    verbose=True,
)

writer = Agent(
    role="Investment Committee Writer",
    goal="Turn the research into a concise and logically structured investment report for a senior stock analyst.",
    backstory=(
        "You write with Barbara Minto's Pyramid Principle discipline: lead with the answer, group ideas logically, "
        "make sections MECE where possible, and communicate in a top-down structure with crisp supporting points."
    ),
    llm=openrouter_llm,
    verbose=True,
)

benchmark_agent = Agent(
    role="Benchmark Analyst",
    goal="Evaluate any index in the list and explain its status relative to what the investor can reasonably know, tolerate, and act on.",
    backstory=(
        "You combine Buffett-style patience with benchmark awareness: you do not value an index like a company; "
        "instead you judge whether the broad market setup is favorable relative to investor knowledge, time horizon, temperament, and opportunity cost."
    ),
    llm=openrouter_llm,
    verbose=True,
)

dcf_agent = Agent(
    role="DCF Analyst",
    goal="Estimate intrinsic value using conservative DCF or DCF-like logic and present a margin-of-safety view.",
    backstory=(
        "You think like Aswath Damodaran at NYU Stern: explicit assumptions, disciplined cash-flow logic, sober discount rates, "
        "base-bear-bull scenarios, humility about uncertainty, and no false precision."
    ),
    llm=openrouter_llm,
    verbose=True,
)

critic = Agent(
    role="Chief Investment Officer",
    goal="Refine the draft for judgment, rigor, clarity, and disciplined capital allocation.",
    backstory=(
        "You think like Charlie Munger: rational, multidisciplinary, skeptical, alert to incentives and second-order effects, "
        "and focused on avoiding stupidity before chasing brilliance. You challenge weak reasoning and force clarity on confidence and downside."
    ),
    llm=openrouter_llm,
    verbose=True,
)

task1 = Task(
    description="""
Investigate the following financial packet using the prepared ticker data as the primary source:

{topic}

For each focus ticker:
- identify the most important signals across business quality, growth, margins, valuation, estimate trend, ownership, catalysts, and risk,
- think with Buffett-style temperament: focus on the business, what it earns, what it may earn, whether the moat is durable, and whether the downside is limited,
- use live web search only to enrich with current context,
- prefer web information from the last 14 days; avoid sources older than 30 days unless the older source is uniquely important,
- clearly separate facts, interpretation, and missing information.
""",
    expected_output="A structured research summary covering each focus ticker and relevant context.",
    agent=researcher,
)

task2 = Task(
    description="""
Take the research and turn it into a structured investment report.

Requirements:
- use Barbara Minto's Pyramid Principle,
- lead with the answer first,
- group points logically and avoid overlap,
- compare each ticker with the benchmark where useful,
- focus on implications, not just facts,
- include thesis, risks, and what to monitor.
""",
    expected_output="A structured investment committee draft with separate sections per ticker.",
    agent=writer,
)

task3 = Task(
    description="""
If any index is present in the input, analyze it using a benchmark lens.

Requirements:
- explain the index status relative to what the investor knows, can tolerate, and can reasonably do with the information,
- do not treat the index like an operating company,
- focus on regime, breadth, risk appetite, and whether the index is favorable from a long-term investor perspective,
- when using web enrichment, prefer market context from the last 14 days and avoid stale commentary older than 30 days unless necessary,
- if no index is present, return a short note saying no benchmark-specific work was needed.
""",
    expected_output="A benchmark analysis note only when an index exists in the input.",
    agent=benchmark_agent,
)

task4 = Task(
    description="""
Estimate intrinsic value for each operating company using conservative DCF or DCF-like logic.

Requirements:
- think like Aswath Damodaran,
- use explicit assumptions and avoid false precision,
- give base, bear, and bull case assumptions,
- state whether the stock appears to trade at a discount or premium to intrinsic value,
- state whether the margin of safety is sufficient,
- be explicit where data quality limits the confidence of the valuation,
- if using external web context, prefer very recent information from the last 14 days and avoid sources older than 30 days unless they are essential,
- if the asset is an index, say DCF does not apply and skip it.
""",
    expected_output="A DCF and margin-of-safety memo for each operating company, plus a note when DCF is not applicable.",
    agent=dcf_agent,
)

task5 = Task(
    description="""
Review the combined draft and improve it.

Requirements:
1. Think like Charlie Munger: attack weak logic, demand clarity, and avoid overconfidence.
2. Remove filler, repetition, and vague claims.
3. Keep the report concise and useful for an experienced analyst.
3a. If external web enrichment is used, prefer sources from the last 14 days and avoid sources older than 30 days unless essential.
4. Return plain text only.
5. Add a confidence score from 0 to 100 for each ticker conclusion.
6. Apply guardrails: if confidence is below 60, explicitly say the case is not strong enough for a high-conviction view.
7. Use this exact structure:

MARKET SNAPSHOT
- short bullets on the benchmark and broad setup

BENCHMARK VIEW
- only if an index exists; otherwise omit this section

ONE SECTION PER FOCUS TICKER
- bullets covering business quality, scenarios, intrinsic value, margin of safety, confidence score, risks, and what to monitor

DATA CAVEATS
- bullets

DATA SOURCE
- Yahoo Finance

NEXT IMPROVEMENTS
- bullets on how to improve the system further
""",
    expected_output="A final plain-text investment report under 4000 characters with confidence scores and guardrails.",
    agent=critic,
)

crew = Crew(
    agents=[researcher, writer, benchmark_agent, dcf_agent, critic],
    tasks=[task1, task2, task3, task4, task5],
    process=Process.sequential,
    verbose=True,
)

try:
    print("Running financial analyst swarm...")
    result = crew.kickoff(inputs={"topic": swarm_payload})
    final_report = getattr(result, "raw", str(result))

    with open(OUTDIR / "final_swarm_report.txt", "w") as f:
        f.write(final_report)

    if SEND_TELEGRAM_FINAL_SUMMARY:
        message_content = "**DAILY EQUITY INTELLIGENCE REPORT**\n\n" + final_report
        for chunk in chunk_text(message_content, MAX_TELEGRAM_CHARS):
            send_to_phone(chunk)

    print("✅ Success! Report generated and Telegram delivery attempted.")
    print(final_report)
except Exception as e:
    print(f"\n❌ CRITICAL ERROR LOGGED:\n{str(e)}")
    send_to_phone(f"Agent execution error: {str(e)}")

Telegram status: 200
Running financial analyst swarm...


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6eadb0db-7a10-4b83-ab13-f670cf4f5ed9                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Equity Research Analyst                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Investigate the following financial packet using the prepared ticker data as the primary source:               │
│                                                                                                                 │
│  {                                                                                                              │
│    "as_of": "2026-06-16",                                                                                       │
│    "benchmark": "^GSPC",                                                                                        │
│    "focus_tickers": [                                                                                           │
│      "NVO",                                                                                                     │
│      "BKR",                                                                                                     │
│      "GEV",                                                                                                     │
│      "GOOG"                                                                                                     │
│    ],                                                                                                           │
│    "index_tickers": [],                                                                                         │
│    "packets": {                                                                                                 │
│      "NVO": {                                                                                                   │
│        "ticker": "NVO",                                                                                         │
│        "asset_type": "equity",                                                                                  │
│        "company_name": "Novo Nordisk A/S",                                                                      │
│        "profile": {                                                                                             │
│          "company_name": "Novo Nordisk A/S",                                                                    │
│          "sector": "Healthcare",                                                                                │
│          "industry": "Drug Manufacturers - General",                                                            │
│          "country": "Denmark"                                                                                   │
│        },                                                                                                       │
│        "market": {                                                                                              │
│          "price": 43.880001068115234,                                                                           │
│          "returns": {                                                                                           │
│            "1m": -0.0679694273254825,                                                                           │
│            "3m": 0.15595369142314142,                                                                           │
│            "6m": -0.10540261466587408,                                                                          │
│            "1y": -0.4426521123188697                   

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Equity Research Analyst                                                                          │
│                                                                                                                 │
│  Thought: Thought: I need to enrich the financial packet data with current web context for each ticker (NVO,    │
│  BKR, GEV, GOOG). Let me search for recent news and developments for each company.                              │
│                                                                                                                 │
│  Using Tool: Search the internet with Serper                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Officer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Let me carefully review the combined draft and apply Munger-like scrutiny throughout.                          │
│                                                                                                                 │
│  **Issues I identify in the current draft:**                                                                    │
│                                                                                                                 │
│  1. **The DCF section for NVO is internally contradictory** — it first calculates $290/share, catches itself,   │
│  recalibrates to $79.50 base case, and never clearly explains the data inconsistency. The "current normalized   │
│  earnings power ~$20B" is pulled from thin air by working backward from market cap. This is sloppy logic that   │
│  should be called out clearly, not buried.                                                                      │
│                                                                                                                 │
│  2. **NVO's debt is under-flagged.** $104.5B net debt on $194.5B market cap is alarming — that's over 50% of    │
│  market cap. The draft treats it as a moderate risk; Munger would say it's a central question: Can the          │
│  business service this debt if GLP-1 pricing pressure persists 3-4 years?                                       │
│                                                                                                                 │
│  3. **GEV's "buffett-style assessment" in the first section says "below $600" entry point, but DCF says base    │
│  case is $489.** These two numbers need reconciliation. If base case fair value is $489, saying $600 is an      │
│  entry point implies only a 23% margin of safety — thin for a Munger-style investment.                          │
│                                                                                                                 │
│  4. **GOOG's DCF math is wrong.** The draft says terminal value CAGR of ~$289/share "in perpetuity" from Year   │
│  10, but then the Bull case goes to $426. The sensitivity table shows the current price implies 3.5% terminal   │
│  growth and 7.5% WACC — but the draft doesn't clearly state whether this is reasonable or not. Given GOOG's     │
│  quality, the market's assumptions may be fair, which undermines the "no margin of safety" conclusion unless    │
│  the analyst explicitly defends why they disagree.                                                              │
│                                                                                                                 │
│  5. **The benchmark section is verbose and non-actionable.** Phrases like "the index is a mirror reflecting a   │
│  market at a crossroads" are filler. Delete.                                                                    │
│                                                                                                                 │
│  6. **The report is massively long** — well over 4000 characters. Needs to be compressed ruthlessly.            │
│                                                        

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 82d947a5-fdb5-4395-b7a1-f29f5a2f7848                                                                     │
│  Agent: Chief Investment Officer                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6eadb0db-7a10-4b83-ab13-f670cf4f5ed9                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output:                                                                                                  │
│                                                                                                                 │
│  Let me carefully review the combined draft and apply Munger-like scrutiny throughout.                          │
│                                                                                                                 │
│  **Issues I identify in the current draft:**                                                                    │
│                                                                                                                 │
│  1. **The DCF section for NVO is internally contradictory** — it first calculates $290/share, catches itself,   │
│  recalibrates to $79.50 base case, and never clearly explains the data inconsistency. The "current normalized   │
│  earnings power ~$20B" is pulled from thin air by working backward from market cap. This is sloppy logic that   │
│  should be called out clearly, not buried.                                                                      │
│                                                                                                                 │
│  2. **NVO's debt is under-flagged.** $104.5B net debt on $194.5B market cap is alarming — that's over 50% of    │
│  market cap. The draft treats it as a moderate risk; Munger would say it's a central question: Can the          │
│  business service this debt if GLP-1 pricing pressure persists 3-4 years?                                       │
│                                                                                                                 │
│  3. **GEV's "buffett-style assessment" in the first section says "below $600" entry point, but DCF says base    │
│  case is $489.** These two numbers need reconciliation. If base case fair value is $489, saying $600 is an      │
│  entry point implies only a 23% margin of safety — thin for a Munger-style investment.                          │
│                                                                                                                 │
│  4. **GOOG's DCF math is wrong.** The draft says terminal value CAGR of ~$289/share "in perpetuity" from Year   │
│  10, but then the Bull case goes to $426. The sensitivity table shows the current price implies 3.5% terminal   │
│  growth and 7.5% WACC — but the draft doesn't clearly state whether this is reasonable or not. Given GOOG's     │
│  quality, the market's assumptions may be fair, which undermines the "no margin of safety" conclusion unless    │
│  the analyst explicitly defends why they disagree.                                                              │
│                                                                                                                 │
│  5. **The benchmark section is verbose and non-actionable.** Phrases like "the index is a mirror reflecting a   │
│  market at a crossroads" are filler. Delete.                                                                    │
│                                                                                                                 │
│  6. **The report is massively long** — well over 4000 

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): Telegram status: 200
Telegram status: 200
Telegram status: 200
Telegram status: 200
✅ Success! Report generated and Telegram delivery attempted.


Let me carefully review the combined draft and apply Munger-like scrutiny throughout.

**Issues I identify in the current draft:**

1. **The DCF section for NVO is internally contradictory** — it first calculates $290/share, catches itself, recalibrates to $79.50 base case, and never clearly explains the data inconsistency. The "current normalized earnings power ~$20B" is pulled from thin air by working backward from market cap. This is sloppy logic that should be called out clearly, not buried.

2. **NVO's debt is under-flagged.** $104.5B net debt on $194.5B market cap is alarming — that's over 50% of market cap. The draft treats it as a moderate risk; Munger would say it's a central question: Can the business service this debt if GLP-1 pricing pressure persists 3-4 years?

